# RetailPulse -- LSTM Demand Forecasting with PyTorch

**Objective:** Build an LSTM-based demand forecasting model using PyTorch. Use sliding window sequences from daily sales data, train with early stopping, evaluate on the same 30-day test set as Prophet, and compare performance.


In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}")
print(f"Device: {device}")

torch.manual_seed(42)
np.random.seed(42)


PyTorch 2.2.2
Device: cpu


In [2]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")


## Load and Prepare Data

In [3]:
lstm_df = pd.read_csv(os.path.join(PROCESSED_DIR, "lstm_ready.csv"), parse_dates=["Date"])
lstm_df = lstm_df.sort_values("Date").reset_index(drop=True)

print(f"Shape: {lstm_df.shape}")
print(f"Date range: {lstm_df['Date'].min().date()} to {lstm_df['Date'].max().date()}")
print(f"Features: {list(lstm_df.columns[1:])}")
lstm_df.head()


Shape: (374, 8)
Date range: 2009-12-01 to 2010-12-09
Features: ['total_revenue', 'total_quantity', 'transaction_count', 'unique_customers', 'day_of_week', 'is_weekend', 'month']


,Date,total_revenue,total_quantity,transaction_count,unique_customers,day_of_week,is_weekend,month
0,2009-12-01,43894.87,24335.0,98.0,91.0,1,0,12
1,2009-12-02,52762.06,29679.0,110.0,94.0,2,0,12
2,2009-12-03,67413.62,48009.0,122.0,106.0,3,0,12
3,2009-12-04,33913.81,19954.0,80.0,76.0,4,0,12
4,2009-12-05,9803.05,5119.0,30.0,26.0,5,1,12


## Feature Scaling

MinMaxScaler normalizes features to [0, 1] range. This is important for LSTM convergence since the network uses sigmoid/tanh activations internally.


In [4]:
feature_cols = ["total_revenue", "total_quantity", "transaction_count",
                "unique_customers", "day_of_week", "is_weekend", "month"]
target_col = "total_revenue"

# Scale features
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(lstm_df[feature_cols])

# Keep a separate scaler for the target (for inverse transform)
target_scaler = MinMaxScaler()
target_scaled = target_scaler.fit_transform(lstm_df[[target_col]])

print(f"Scaled data shape: {scaled_data.shape}")
print(f"Value range: [{scaled_data.min():.4f}, {scaled_data.max():.4f}]")


Scaled data shape: (374, 7)
Value range: [0.0000, 1.0000]


## Sliding Window Sequences

Create input sequences of `LOOKBACK` days to predict the next day's revenue.


In [5]:
LOOKBACK = 30  # use last 30 days to predict next day
TEST_DAYS = 30

class TimeSeriesDataset(Dataset):
    def __init__(self, data, target, lookback):
        self.data = data
        self.target = target
        self.lookback = lookback

    def __len__(self):
        return len(self.data) - self.lookback

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.lookback]
        y = self.target[idx + self.lookback]
        return torch.FloatTensor(x), torch.FloatTensor(y)

# Train/test split (same 30-day split as Prophet)
split_idx = len(scaled_data) - TEST_DAYS

train_data = scaled_data[:split_idx]
train_target = target_scaled[:split_idx]
test_data = scaled_data  # need full data for lookback windows into test
test_target = target_scaled

train_dataset = TimeSeriesDataset(train_data, train_target, LOOKBACK)
test_dataset = TimeSeriesDataset(test_data, test_target, LOOKBACK)

# Only evaluate on the last TEST_DAYS predictions
test_indices = list(range(split_idx - LOOKBACK, len(test_data) - LOOKBACK))

print(f"Lookback window: {LOOKBACK} days")
print(f"Train sequences: {len(train_dataset)}")
print(f"Test evaluation points: {len(test_indices)}")
print(f"Input shape per sample: ({LOOKBACK}, {len(feature_cols)})")


Lookback window: 30 days
Train sequences: 314
Test evaluation points: 30
Input shape per sample: (30, 7)


In [6]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"Train batches: {len(train_loader)}")
print(f"Batch size: {BATCH_SIZE}")


Train batches: 9
Batch size: 32


## LSTM Model Architecture

A 2-layer LSTM with dropout, followed by a fully connected output layer. The model takes a sequence of daily feature vectors and outputs the next day's revenue prediction.


In [7]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x shape: (batch, seq_len, features)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)

        lstm_out, _ = self.lstm(x, (h0, c0))

        # Take the last time step output
        last_output = lstm_out[:, -1, :]
        out = self.dropout(last_output)
        out = self.fc(out)
        return out

INPUT_DIM = len(feature_cols)
HIDDEN_DIM = 64
NUM_LAYERS = 2
OUTPUT_DIM = 1
DROPOUT = 0.2

model = LSTMForecaster(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, OUTPUT_DIM, DROPOUT).to(device)
print(model)
print()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


LSTMForecaster(
  (lstm): LSTM(7, 64, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

Total parameters: 52,033
Trainable parameters: 52,033


## Training

In [8]:
LEARNING_RATE = 0.001
EPOCHS = 150
PATIENCE = 15  # early stopping patience

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5, verbose=False
)

# Training loop with early stopping
train_losses = []
best_loss = float("inf")
patience_counter = 0
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    batch_count = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1

    avg_loss = epoch_loss / batch_count
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)

    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join("..", "models", "lstm_best.pt"))
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or patience_counter == PATIENCE:
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1:>3d}/{EPOCHS}  |  Loss: {avg_loss:.6f}  "
              f"|  Best: {best_loss:.6f} (ep {best_epoch+1})  |  LR: {current_lr:.6f}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nTraining complete. Best epoch: {best_epoch+1}, Best loss: {best_loss:.6f}")


Epoch  10/150  |  Loss: 0.017589  |  Best: 0.016863 (ep 8)  |  LR: 0.001000


Epoch  20/150  |  Loss: 0.012823  |  Best: 0.012823 (ep 20)  |  LR: 0.001000


Epoch  30/150  |  Loss: 0.012192  |  Best: 0.010173 (ep 24)  |  LR: 0.000500


Epoch  39/150  |  Loss: 0.011646  |  Best: 0.010173 (ep 24)  |  LR: 0.000250
Early stopping at epoch 39

Training complete. Best epoch: 24, Best loss: 0.010173


In [9]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_losses, color="#3498db", linewidth=1.2)
ax.axvline(best_epoch, color="#e74c3c", linestyle="--", alpha=0.7,
           label=f"Best epoch ({best_epoch+1})")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("LSTM Training Loss")
ax.legend()
fig.tight_layout()
save_fig(fig, "27_lstm_training_loss.png")
plt.show()


Saved: 27_lstm_training_loss.png


## Evaluation on Test Set

In [10]:
# Load best model
model.load_state_dict(torch.load(os.path.join("..", "models", "lstm_best.pt"),
                                  weights_only=True))
model.eval()

# Generate predictions for test period
test_predictions = []
test_actuals = []

with torch.no_grad():
    for idx in test_indices:
        x = torch.FloatTensor(test_data[idx : idx + LOOKBACK]).unsqueeze(0).to(device)
        pred = model(x).cpu().numpy().flatten()
        test_predictions.append(pred[0])
        test_actuals.append(test_target[idx + LOOKBACK][0])

# Inverse transform to original scale
test_predictions = np.array(test_predictions).reshape(-1, 1)
test_actuals = np.array(test_actuals).reshape(-1, 1)

pred_original = target_scaler.inverse_transform(test_predictions).flatten()
actual_original = target_scaler.inverse_transform(test_actuals).flatten()

# Get corresponding dates
test_dates = lstm_df["Date"].iloc[split_idx:split_idx + len(pred_original)].values

print(f"Test predictions: {len(pred_original)}")


Test predictions: 30


In [11]:
# Compute metrics
non_zero = actual_original != 0
mape = np.mean(np.abs((actual_original[non_zero] - pred_original[non_zero])
                       / actual_original[non_zero])) * 100
mae = np.mean(np.abs(actual_original - pred_original))
rmse = np.sqrt(np.mean((actual_original - pred_original) ** 2))

print("LSTM Model Metrics (Test Set):")
print(f"  MAPE:  {mape:.2f}%")
print(f"  MAE:   {mae:,.2f}")
print(f"  RMSE:  {rmse:,.2f}")


LSTM Model Metrics (Test Set):
  MAPE:  20.41%
  MAE:   11,514.43
  RMSE:  15,097.07


In [12]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(test_dates, actual_original, color="#2c3e50", linewidth=1.5,
        label="Actual", marker="o", markersize=3)
ax.plot(test_dates, pred_original, color="#e74c3c", linewidth=1.5,
        label="LSTM Predicted", marker="s", markersize=3)
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
ax.set_title(f"LSTM Forecast -- Test Set (MAPE: {mape:.2f}%)")
ax.legend()
fig.tight_layout()
save_fig(fig, "28_lstm_forecast.png")
plt.show()


Saved: 28_lstm_forecast.png


## Residual Analysis

In [13]:
residuals = actual_original - pred_original

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residual plot
axes[0].scatter(pred_original, residuals, alpha=0.6, color="#3498db", s=30)
axes[0].axhline(0, color="#e74c3c", linestyle="--")
axes[0].set_xlabel("Predicted Revenue")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs Predicted")

# Residual histogram
axes[1].hist(residuals, bins=20, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="#e74c3c", linestyle="--")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution")

# Actual vs Predicted
min_val = min(actual_original.min(), pred_original.min())
max_val = max(actual_original.max(), pred_original.max())
axes[2].scatter(actual_original, pred_original, alpha=0.6, color="#9b59b6", s=30)
axes[2].plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5, label="Perfect fit")
axes[2].set_xlabel("Actual Revenue")
axes[2].set_ylabel("Predicted Revenue")
axes[2].set_title("Actual vs Predicted")
axes[2].legend()

fig.suptitle("LSTM Residual Analysis", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "29_lstm_residuals.png")
plt.show()


Saved: 29_lstm_residuals.png


In [14]:
# Save LSTM predictions
lstm_results = pd.DataFrame({
    "ds": test_dates,
    "actual": actual_original,
    "lstm_predicted": pred_original,
    "residual": residuals,
})
lstm_path = os.path.join(PROCESSED_DIR, "lstm_predictions.csv")
lstm_results.to_csv(lstm_path, index=False)
print(f"Saved LSTM predictions: {lstm_path}")
lstm_results.head()


Saved LSTM predictions: ../data/processed/lstm_predictions.csv


,ds,actual,lstm_predicted,residual
0,2010-11-10,73575.93,42459.105469,31116.824531
1,2010-11-11,59712.61,38196.585938,21516.024063
2,2010-11-12,36167.25,30847.935547,5319.314453
3,2010-11-13,0.00,23126.482422,-23126.482422
4,2010-11-14,27934.71,22354.996094,5579.713906


## Summary

In [15]:
print("LSTM DEMAND FORECASTING COMPLETE")
print("=" * 55)
print()
print("Architecture:")
print(f"  Input:  {INPUT_DIM} features x {LOOKBACK} days lookback")
print(f"  LSTM:   {NUM_LAYERS} layers, {HIDDEN_DIM} hidden units")
print(f"  Dropout: {DROPOUT}")
print(f"  Params: {total_params:,}")
print()
print("Training:")
print(f"  Epochs run: {len(train_losses)} (early stopped)")
print(f"  Best epoch: {best_epoch + 1}")
print(f"  Best loss:  {best_loss:.6f}")
print()
print("Test Performance:")
print(f"  MAPE:  {mape:.2f}%")
print(f"  MAE:   {mae:,.2f}")
print(f"  RMSE:  {rmse:,.2f}")
print()
print("Exported files:")
print(f"  models/lstm_best.pt        (best model weights)")
print(f"  lstm_predictions.csv       (test set predictions)")
print()
print("Next steps:")
print("  - Hybrid Prophet + LSTM ensemble")
print("  - Compare and select final forecasting model")


LSTM DEMAND FORECASTING COMPLETE

Architecture:
  Input:  7 features x 30 days lookback
  LSTM:   2 layers, 64 hidden units
  Dropout: 0.2
  Params: 52,033

Training:
  Epochs run: 39 (early stopped)
  Best epoch: 24
  Best loss:  0.010173

Test Performance:
  MAPE:  20.41%
  MAE:   11,514.43
  RMSE:  15,097.07

Exported files:
  models/lstm_best.pt        (best model weights)
  lstm_predictions.csv       (test set predictions)

Next steps:
  - Hybrid Prophet + LSTM ensemble
  - Compare and select final forecasting model
